# PD Model Training – Quantum (QSVM / VQC)

**Purpose:** Train hybrid quantum-classical PD models on the same engineered LendingClub features as the XGBoost notebook. We use **PennyLane** (or Qiskit) simulators only (CPU, no real hardware). Models: **QSVM** (quantum kernel) and/or **VQC** (Variational Quantum Classifier / QNN). Compare AUC-ROC, F1, and KS-drift vs the classical XGBoost baseline.

**Input:** `data/credit_risk_pd/LendingClub/processed/lendingclub_engineered.parquet` (run notebook 01 first).

**Dependencies:** `pennylane`, `pennylane-sf` (or default.qubit); optionally `qiskit-aer` for Qiskit path. Install: `pip install pennylane pennylane-sf`.

## 1. Load engineered data (same as notebook 02)

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Find repo root (works when run from repo root or from notebooks/)
ROOT = Path.cwd()
for _ in range(5):
    if (ROOT / "credit_risk").is_dir() and (ROOT / "data").is_dir():
        break
    ROOT = ROOT.parent
if not (ROOT / "credit_risk").is_dir():
    raise RuntimeError('Repo root not found. Run this notebook from the ocr-agentic-rag folder (or notebooks/ subfolder).')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from credit_risk.feature_engineering.common_features import get_feature_names

DATA_PATH = ROOT / "data" / "credit_risk_pd" / "LendingClub" / "processed" / "lendingclub_engineered.parquet"
if not DATA_PATH.exists():
    raise FileNotFoundError("Run 01_lendingclub_feature_engineering.ipynb first.")

df = pd.read_parquet(DATA_PATH)
feature_names = get_feature_names()
X = df[[c for c in feature_names if c in df.columns]].astype(float)
y = df["default"].values
if X.shape[1] != len(feature_names):
    for c in feature_names:
        if c not in X.columns:
            X[c] = 0.0
X = X[feature_names].values

# Scale for quantum circuits (angles / kernel inputs)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
# Reduce dimension for NISQ (e.g. 4–6 qubits = 4–6 features) to keep runtime tractable
n_features_q = min(6, X_scaled.shape[1])
X_q = X_scaled[:, :n_features_q]

# Same split for quantum (6 feat) and classical (15 feat) comparison
train_idx, val_idx = train_test_split(np.arange(len(y)), test_size=0.2, random_state=42, stratify=y)
X_train, X_val = X_q[train_idx], X_q[val_idx]
y_train, y_val = y[train_idx], y[val_idx]
X_full = df[feature_names].astype(float).values
X_val_full = X_full[val_idx]
print(f"Train {X_train.shape[0]:,} / Val {X_val.shape[0]:,} | Features for quantum: {n_features_q}")

## 2. PennyLane – Variational Quantum Classifier (VQC)

We use a simple angle-encoding + variational layer and a binary observable. Run on `default.qubit` (CPU simulator).

In [2]:
try:
    import pennylane as qml
    from pennylane import numpy as pnp
    PENNYLANE_AVAILABLE = True
except ImportError:
    PENNYLANE_AVAILABLE = False
    print("Install: pip install pennylane")

if PENNYLANE_AVAILABLE:
    n_qubits = n_features_q
    dev = qml.device("default.qubit", wires=n_qubits)

    @qml.qnode(dev)
    def circuit(params, x):
        for i in range(n_qubits):
            qml.RY(x[i] * np.pi, wires=i)
        for i in range(n_qubits):
            qml.RZ(params[i], wires=i)
        for i in range(n_qubits - 1):
            qml.CNOT(wires=[i, i + 1])
        for i in range(n_qubits):
            qml.RZ(params[n_qubits + i], wires=i)
        return qml.expval(qml.PauliZ(0))

    def vqc_predict(params, X_batch):
        return np.array([circuit(params, x) for x in X_batch])

    # Simple training loop (gradient descent on binary cross-entropy proxy)
    from scipy.optimize import minimize
    def loss(params):
        pred = np.array([circuit(params, x) for x in X_train[:200]])  # subsample for speed
        pred = (pred + 1) / 2  # map [-1,1] -> [0,1]
        pred = np.clip(pred, 1e-6, 1 - 1e-6)
        return -np.mean(y_train[:200] * np.log(pred) + (1 - y_train[:200]) * np.log(1 - pred))

    init = pnp.random.uniform(0, 2 * np.pi, size=2 * n_qubits)
    res = minimize(loss, init, method="COBYLA", options={"maxiter": 50})
    params_vqc = res.x
    print('VQC training finished. Params:', params_vqc[:4], '...')
else:
    params_vqc = None

Install: pip install pennylane


In [3]:
from sklearn.metrics import roc_auc_score, f1_score

def eval_binary(y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)
    return {
        "auc_roc": roc_auc_score(y_true, y_prob),
        "f1": f1_score(y_true, y_pred, zero_division=0),
    }

if PENNYLANE_AVAILABLE and params_vqc is not None:
    pred_val = np.array([circuit(params_vqc, x) for x in X_val])
    pred_val = (pred_val + 1) / 2
    results_vqc = eval_binary(y_val, pred_val)
    print('VQC (PennyLane) validation:', results_vqc)
else:
    results_vqc = {"auc_roc": 0.5, "f1": 0.0}
    print('VQC skipped (PennyLane not available or training failed).')

VQC skipped (PennyLane not available or training failed).


## 3. Classical baseline comparison (same split)

Load the XGBoost model from notebook 02 (or train a quick XGBoost on the same 6 features) to report AUC/F1 comparison and uplift %.

In [4]:
import joblib

model_path = ROOT / "models" / "pd" / "pd_model_local_v1.pkl"
if model_path.exists():
    data = joblib.load(model_path)
    xgb_model = data["model"]
    # XGBoost was trained on full 15 features
    p_xgb = xgb_model.predict_proba(X_val_full)[:, 1]
    results_xgb = eval_binary(y_val, p_xgb)
    print('XGBoost (classical, 15-feat) validation:', results_xgb)
else:
    # Quick XGBoost on same 6 features
    import xgboost as xgb
    scale = (y_train == 0).sum() / max((y_train == 1).sum(), 1)
    clf = xgb.XGBClassifier(max_depth=4, n_estimators=50, scale_pos_weight=scale, random_state=42)
    clf.fit(X_train, y_train)
    p_xgb = clf.predict_proba(X_val)[:, 1]
    results_xgb = eval_binary(y_val, p_xgb)
    print('XGBoost (6-feature baseline) validation:', results_xgb)

print('\nUplift vs classical (VQC - XGB): AUC', round(results_vqc['auc_roc'] - results_xgb['auc_roc'], 4), 'F1', round(results_vqc['f1'] - results_xgb['f1'], 4))

ModuleNotFoundError: No module named 'xgboost'

## 4. Save quantum model artifacts (optional)

Persist VQC parameters and scaler so that evaluation or a small inference script can run quantum PD predictions. NISQ note: in production, consider noise mitigation and qubit scaling; simulators are for prototyping.

In [ ]:
MODEL_DIR = ROOT / "models" / "pd"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
if PENNYLANE_AVAILABLE and params_vqc is not None:
    joblib.dump({
        "params": params_vqc,
        "n_qubits": n_qubits,
        "scaler": scaler,
        "feature_names": feature_names[:n_features_q],
        "backend": "pennylane.default.qubit",
    }, MODEL_DIR / "pd_quantum_vqc_v1.pkl")
    print('Saved VQC params to models/pd/pd_quantum_vqc_v1.pkl')
print('Done. Use eval_runner with PDModel for classical; extend run_model for quantum backend if needed.')